# 🔮 AirSense — Machine Learning Model Evaluation & Diagnostics
**MSc Business Intelligence & Analytics | Capstone Project (Phase 3: Weeks 5 & 6)**

This notebook evaluates the **LightGBM Gradient Boosting AQI Forecasting Model**.

### Core Analytical Objectives:
1. **Walk-Forward Validation**: Time-series cross-validation without lookahead bias.
2. **Baseline Benchmarking**: Comparing against Naive Persistence (24h lag baseline).
3. **Error Metrics**: Evaluating RMSE, MAE, R², and Directional Accuracy.
4. **Feature Importance**: Identifying key physical, temporal, and autoregressive drivers.
5. **Forecast Visualisation**: Visualising 48-hour forward projection with confidence intervals.

In [ ]:
import sys
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.config import load_settings
from consumer.db import create_db_engine
from ml.training.features import create_features, FEATURE_COLUMNS, TARGET_COLUMN

settings = load_settings(PROJECT_ROOT / "config.yaml")
engine = create_db_engine(settings)
print("✅ Connected to TimescaleDB!")

## 1. Load Champion Model Artifacts & MLflow Metrics

In [ ]:
metrics_path = PROJECT_ROOT / "ml" / "production" / "metrics.json"
model_path = PROJECT_ROOT / "ml" / "production" / "lgbm_aqi.pkl"

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

model = joblib.load(model_path)
print(f"Champion Model: {metrics.get('model_version')}")
print(f"Walk-Forward CV Mean RMSE: {metrics.get('rmse'):.2f} AQI units (Goal < 30.0)")
print(f"Walk-Forward CV Mean MAE:  {metrics.get('mae'):.2f} AQI units")
print(f"Variance Explained (R²):   {metrics.get('r2'):.3f}")
print(f"Directional Accuracy:      {metrics.get('directional_accuracy'):.1f}%")

## 2. Feature Importance Analysis (Gradient Boosted Splits & Gain)

In [ ]:
# Extract feature importance from the trained LightGBM booster
importance = model.feature_importances_
feat_names = FEATURE_COLUMNS

feat_imp_df = pd.DataFrame({
    "Feature": feat_names,
    "Importance": importance,
}).sort_values(by="Importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df["Feature"], feat_imp_df["Importance"], color="#2563EB", edgecolor="#1E40AF")
plt.title("AirSense LightGBM Feature Importance (Split Count)", fontsize=14, fontweight="bold")
plt.xlabel("Relative Importance (Tree Splits)")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 3. Naive Baseline vs. LightGBM Benchmark Comparison

In [ ]:
# Load historical data
query = text("SELECT time, city, aqi, pm25, pm10, no2 FROM aqi_readings ORDER BY time ASC;")
with engine.connect() as conn:
    df_raw = pd.read_sql(query, conn)

df_feat = create_features(df_raw, is_training=True)

# Naive persistence baseline: predict AQI(t) = AQI(t - 24h)
naive_pred = df_feat["aqi_lag_24h"]
actual = df_feat["aqi"]

naive_rmse = np.sqrt(np.mean((actual - naive_pred) ** 2))
naive_mae = np.mean(np.abs(actual - naive_pred))

print("=== BENCHMARK COMPARISON ===")
print(f"Naive 24h Persistence:  RMSE = {naive_rmse:.2f} AQI | MAE = {naive_mae:.2f} AQI")
print(f"LightGBM Model:        RMSE = {metrics['rmse']:.2f} AQI | MAE = {metrics['mae']:.2f} AQI")
pct_improvement = ((naive_rmse - metrics['rmse']) / naive_rmse) * 100
print(f"✨ LightGBM Error Reduction vs. Baseline: {pct_improvement:.1f}%")

## 4. Visualising Stored 48-Hour Forecasts vs. Actuals

In [ ]:
# Query forecasts for Ahmedabad
fc_query = text("""
    SELECT forecast_time, predicted_aqi, generated_at
    FROM aqi_forecasts
    WHERE city = 'ahmedabad'
    ORDER BY forecast_time ASC
    LIMIT 48;
""")
with engine.connect() as conn:
    fc_df = pd.read_sql(fc_query, conn)

print(f"Ahmedabad 48h Forecasts: {len(fc_df)} hourly steps")
print(f"Projected Range: Min={fc_df['predicted_aqi'].min():.1f} | Max={fc_df['predicted_aqi'].max():.1f} AQI")

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(fc_df["forecast_time"]), fc_df["predicted_aqi"], color="#EA580C", linestyle="--", marker="o", label="LightGBM 48h Forecast")
plt.title("AirSense 48-Hour Forecast Trajectory — Ahmedabad", fontsize=13, fontweight="bold")
plt.xlabel("Forecast Horizon (UTC)")
plt.ylabel("Predicted AQI")
plt.axhline(100, color="#F59E0B", linestyle=":", label="Moderate AQI (100)")
plt.axhline(200, color="#EF4444", linestyle=":", label="Poor AQI (200)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()